In [12]:
import csv
import json
import os
import shutil
from collections import Counter

SURNAME = "Solina"
STUDENT_ID = "TUPM-26-2302"
SEED_DIGIT = 2
ID_SUM = 15

MAX_RECORDS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

import glob
import os

matches = glob.glob(r"C:\Users\James Solina\**\weblog.csv", recursive=True)
INPUT_CSV = matches[0] if matches else "weblog.csv"
print("Using dataset path:", INPUT_CSV)
FILTERED_CSV = "filtered_weblog_4xx.csv"
SUMMARY_JSON = "weblog_summary.json"
BACKUP_JSON = "weblog_summary_backup.json"


def parse_status_code(raw_status):
    if raw_status is None:
        return None
    cleaned = str(raw_status).strip()
    if cleaned.isdigit() and len(cleaned) == 3:
        return int(cleaned)
    return None


def main():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Dataset '{INPUT_CSV}' not found.")
        return

    records_processed = 0
    invalid_records = 0
    status_counter = Counter()
    url_counter = Counter()
    selected_rows = []

    with open(INPUT_CSV, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
        reader = csv.DictReader(infile)
        for row in reader:
            if records_processed >= MAX_RECORDS:
                break

            ip = (row.get("IP") or "").strip()
            timestamp = (row.get("Time") or "").strip()
            url = (row.get("URL") or "").strip()
            raw_status = row.get("Staus") if "Staus" in row else row.get("Status")
            status_code = parse_status_code(raw_status)

            records_processed += 1

            if not ip or not url or status_code is None:
                invalid_records += 1
                continue

            if status_code // 100 == TARGET_STATUS_CLASS:
                status_counter[str(status_code)] += 1
                url_counter[url] += 1
                selected_rows.append({
                    "IP": ip,
                    "Time": timestamp,
                    "URL": url,
                    "Staus": status_code
                })

    top_5_urls = [
        {"url": url, "count": count}
        for url, count in url_counter.most_common(5)
    ]

    with open(FILTERED_CSV, mode="w", encoding="utf-8", newline="") as outfile:
        fieldnames = ["IP", "Time", "URL", "Staus"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(selected_rows)

    summary_data = {
        "student_info": {
            "surname": SURNAME,
            "student_id": STUDENT_ID,
            "seed_digit": SEED_DIGIT,
            "id_sum": ID_SUM
        },
        "records_processed": records_processed,
        "invalid_or_missing_records": invalid_records,
        "target_status_class": f"{TARGET_STATUS_CLASS}xx",
        "records_in_target_class": len(selected_rows),
        "count_per_status_code": dict(sorted(status_counter.items())),
        "five_most_requested_urls": top_5_urls,
        "filtered_csv_file": FILTERED_CSV
    }

    with open(SUMMARY_JSON, mode="w", encoding="utf-8") as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)

    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
    backup_verified = os.path.exists(BACKUP_JSON) and os.path.getsize(BACKUP_JSON) > 0

    print("=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===")
    print(f"Student                 : {SURNAME} ({STUDENT_ID})")
    print(f"Records processed       : {records_processed}")
    print(f"Target status class     : {TARGET_STATUS_CLASS}xx")
    print(f"Records in target class : {len(selected_rows)}")
    print(f"Count per status code   : {dict(sorted(status_counter.items()))}")
    print("Five most requested URLs:")
    if top_5_urls:
        for idx, item in enumerate(top_5_urls, start=1):
            print(f"  {idx}. {item['url']} ({item['count']} requests)")
    else:
        print("  None found in target class.")
    print(f"Filtered CSV file       : {FILTERED_CSV}")
    print(f"JSON summary file       : {SUMMARY_JSON}")
    print(f"Backup verification     : {'Verified (' + BACKUP_JSON + ')' if backup_verified else 'Failed'}")
main()

Using dataset path: C:\Users\James Solina\Downloads\archive\weblog.csv
=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===
Student                 : Solina (TUPM-26-2302)
Records processed       : 1015
Target status class     : 4xx
Records in target class : 5
Count per status code   : {'404': 5}
Five most requested URLs:
  1. GET /robots.txt HTTP/1.1 (4 requests)
  2. GET /favicon.ico HTTP/1.1 (1 requests)
Filtered CSV file       : filtered_weblog_4xx.csv
JSON summary file       : weblog_summary.json
Backup verification     : Verified (weblog_summary_backup.json)


In [18]:
import csv
import json
import os
import shutil

SURNAME = "Solina"
STUDENT_ID = "TUPM-26-2302"
SEED_DIGIT = 2

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

import glob
matches = glob.glob(r"C:\Users\James Solina\New folder\**\user_behavior_dataset.csv", recursive=True)
INPUT_CSV = matches[0] if matches else "user_behavior_dataset.csv"
EXPORTED_CSV = "filtered_mobile_usage_android_class3.csv"
BACKUP_CSV = "filtered_mobile_usage_android_class3_backup.csv"
SUMMARY_JSON = "mobile_usage_summary.json"

EXPORT_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]


def safe_float(value):
    try:
        return float(str(value).strip())
    except (ValueError, TypeError):
        return None


def safe_int(value):
    try:
        return int(float(str(value).strip()))
    except (ValueError, TypeError):
        return None


def main():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Dataset '{INPUT_CSV}' not found.")
        return

    total_records = 0
    os_matching_records = 0
    selected_users = []
    total_battery = 0.0
    total_data_usage = 0.0

    with open(INPUT_CSV, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
        reader = csv.DictReader(infile)
        for row in reader:
            total_records += 1

            os_name = (row.get("Operating System") or "").strip()
            behavior_class = safe_int(row.get("User Behavior Class"))

            if os_name.lower() == TARGET_OS.lower():
                os_matching_records += 1

                if behavior_class == TARGET_BEHAVIOR_CLASS:
                    battery = safe_float(row.get("Battery Drain (mAh/day)"))
                    data_mb = safe_float(row.get("Data Usage (MB/day)"))

                    if battery is None or data_mb is None:
                        continue

                    total_battery += battery
                    total_data_usage += data_mb

                    selected_users.append({
                        "User ID": (row.get("User ID") or "").strip(),
                        "Device Model": (row.get("Device Model") or "").strip(),
                        "Operating System": os_name,
                        "Battery Drain (mAh/day)": battery,
                        "Data Usage (MB/day)": data_mb
                    })

    selected_count = len(selected_users)
    if selected_count > 0:
        avg_battery = round(total_battery / selected_count, 2)
        avg_data_usage = round(total_data_usage / selected_count, 2)
    else:
        avg_battery = 0.0
        avg_data_usage = 0.0

    with open(EXPORTED_CSV, mode="w", encoding="utf-8", newline="") as outfile:
        writer = csv.DictWriter(outfile, fieldnames=EXPORT_FIELDS)
        writer.writeheader()
        writer.writerows(selected_users)

    exported_row_count = 0
    with open(EXPORTED_CSV, mode="r", encoding="utf-8", newline="") as verifyfile:
        verify_reader = csv.DictReader(verifyfile)
        for _ in verify_reader:
            exported_row_count += 1

    count_verified = exported_row_count == selected_count

    shutil.copy2(EXPORTED_CSV, BACKUP_CSV)
    backup_exists = os.path.exists(BACKUP_CSV)
    backup_size = os.path.getsize(BACKUP_CSV) if backup_exists else 0

    summary_data = {
        "student_info": {
            "surname": SURNAME,
            "student_id": STUDENT_ID,
            "seed_digit": SEED_DIGIT
        },
        "operating_system": TARGET_OS,
        "target_behavior_class": TARGET_BEHAVIOR_CLASS,
        "total_records_in_dataset": total_records,
        "records_matching_operating_system": os_matching_records,
        "records_in_target_behavior_class": selected_count,
        "average_battery_drain_mah_per_day": avg_battery,
        "average_data_usage_mb_per_day": avg_data_usage,
        "count_verification": {
            "selected_records": selected_count,
            "exported_csv_rows": exported_row_count,
            "verified": count_verified
        },
        "exported_csv_file": EXPORTED_CSV,
        "backup_csv_file": BACKUP_CSV,
        "backup_file_size_bytes": backup_size
    }

    with open(SUMMARY_JSON, mode="w", encoding="utf-8") as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)

    print("=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===")
    print(f"Student                           : {SURNAME} ({STUDENT_ID})")
    print(f"Operating system and behavior class: {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
    print(f"Total records in dataset          : {total_records}")
    print(f"Records matching operating system : {os_matching_records}")
    print(f"Records in target behavior class  : {selected_count}")
    print(f"Average battery drain (mAh/day)   : {avg_battery}")
    print(f"Average data usage (MB/day)       : {avg_data_usage}")
    print(f"Count verification                : {'Matched (' + str(exported_row_count) + ' rows)' if count_verified else 'Mismatch'}")
    print(f"Exported CSV file                 : {EXPORTED_CSV}")
    print(f"JSON summary file                 : {SUMMARY_JSON}")
    print(f"Backup file and size              : {BACKUP_CSV} ({backup_size} bytes)")

main()

=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===
Student                           : Solina (TUPM-26-2302)
Operating system and behavior class: Android, Class 3
Total records in dataset          : 700
Records matching operating system : 554
Records in target behavior class  : 111
Average battery drain (mAh/day)   : 1521.69
Average data usage (MB/day)       : 820.26
Count verification                : Matched (111 rows)
Exported CSV file                 : filtered_mobile_usage_android_class3.csv
JSON summary file                 : mobile_usage_summary.json
Backup file and size              : filtered_mobile_usage_android_class3_backup.csv (4512 bytes)


In [26]:
import csv
import json
import os
import shutil
from collections import Counter

SURNAME = "Solina"
STUDENT_ID = "TUPM-26-2302"
SEED_DIGIT = 2

TARGET_YEAR = 2015 + SEED_DIGIT
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

import glob
matches = glob.glob(r"C:\Users\James Solina\New folder\**\Global_Cybersecurity_Threats_2015-2024.csv", recursive=True)
INPUT_CSV = matches[0] if matches else "Global_Cybersecurity_Threats_2015-2024.csv"
FILTERED_CSV = f"filtered_cybersecurity_{TARGET_YEAR}.csv"
SUMMARY_JSON = f"cybersecurity_top5_{TARGET_YEAR}.json"
BACKUP_JSON = f"cybersecurity_top5_{TARGET_YEAR}_backup.json"
REPORT_TXT = f"cybersecurity_report_{TARGET_YEAR}.txt"


def safe_int(value):
    try:
        return int(float(str(value).strip()))
    except (ValueError, TypeError):
        return None


def safe_float(value):
    try:
        cleaned = str(value).strip().replace("$", "").replace(",", "")
        return float(cleaned)
    except (ValueError, TypeError):
        return None


def main():
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Dataset '{INPUT_CSV}' not found.")
        return

    total_records = 0
    selected_incidents = []
    attack_counter = Counter()
    valid_losses = []
    invalid_loss_count = 0
    fieldnames = []

    with open(INPUT_CSV, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames or []

        for idx, row in enumerate(reader):
            total_records += 1
            year_val = safe_int(row.get("Year"))

            if year_val == TARGET_YEAR:
                attack_type = (row.get(COUNT_FIELD) or "Unknown").strip()
                attack_counter[attack_type] += 1

                loss_val = safe_float(row.get(LOSS_FIELD))
                if loss_val is not None:
                    valid_losses.append(loss_val)
                else:
                    invalid_loss_count += 1

                selected_incidents.append((idx, loss_val, row))

    incidents_in_year = len(selected_incidents)
    total_loss = round(sum(valid_losses), 2) if valid_losses else 0.0
    avg_loss = round(total_loss / len(valid_losses), 2) if valid_losses else 0.0

    rankable = [item for item in selected_incidents if item[1] is not None]
    rankable.sort(key=lambda item: (-item[1], item[0]))
    top_5_incidents = [item[2] for item in rankable[:5]]

    with open(FILTERED_CSV, mode="w", encoding="utf-8", newline="") as outfile:
        if fieldnames:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            for _, _, row in selected_incidents:
                writer.writerow(row)

    json_payload = {
        "student_info": {
            "surname": SURNAME,
            "student_id": STUDENT_ID,
            "seed_digit": SEED_DIGIT
        },
        "target_year": TARGET_YEAR,
        "total_records_in_dataset": total_records,
        "incidents_in_target_year": incidents_in_year,
        "invalid_financial_records": invalid_loss_count,
        "count_per_attack_type": dict(sorted(attack_counter.items())),
        "total_financial_loss_usd_millions": total_loss,
        "average_financial_loss_usd_millions": avg_loss,
        "five_costliest_incidents": top_5_incidents
    }

    with open(SUMMARY_JSON, mode="w", encoding="utf-8") as jsonfile:
        json.dump(json_payload, jsonfile, indent=4)

    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
    backup_verified = os.path.exists(BACKUP_JSON) and os.path.getsize(BACKUP_JSON) > 0

    report_lines = [
        "============================================================",
        f"CYBERSECURITY INCIDENT REPORT ({TARGET_YEAR})",
        f"Student: {SURNAME} | Student ID: {STUDENT_ID}",
        "============================================================",
        f"Target Year                     : {TARGET_YEAR}",
        f"Total Records in Dataset        : {total_records}",
        f"Incidents in Target Year        : {incidents_in_year}",
        f"Invalid Financial Loss Entries  : {invalid_loss_count}",
        f"Total Financial Loss ($M)       : {total_loss}",
        f"Average Financial Loss ($M)     : {avg_loss}",
        "",
        "--- Incident Count per Attack Type ---"
    ]

    if attack_counter:
        for attack, count in sorted(attack_counter.items()):
            report_lines.append(f"  - {attack}: {count}")
    else:
        report_lines.append("  No incidents recorded for this year.")

    report_lines.append("")
    report_lines.append("--- Five Costliest Incidents ---")
    if top_5_incidents:
        for rank, inc in enumerate(top_5_incidents, start=1):
            country = inc.get("Country", "N/A")
            attack = inc.get(COUNT_FIELD, "N/A")
            industry = inc.get("Target Industry", "N/A")
            loss = inc.get(LOSS_FIELD, "N/A")
            report_lines.append(
                f"  {rank}. Country: {country} | Attack: {attack} | Industry: {industry} | Loss: ${loss}M"
            )
    else:
        report_lines.append("  No valid financial loss records found.")

    report_lines.extend([
        "",
        "--- Output File Information ---",
        f"  Filtered CSV File  : {FILTERED_CSV}",
        f"  JSON Summary File  : {SUMMARY_JSON}",
        f"  JSON Backup File   : {BACKUP_JSON} (Verified: {backup_verified})",
        f"  Text Report File   : {REPORT_TXT}",
        "============================================================"
    ])

    with open(REPORT_TXT, mode="w", encoding="utf-8") as txtfile:
        txtfile.write("\n".join(report_lines) + "\n")

    print("=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===")
    print(f"Student                        : {SURNAME} ({STUDENT_ID})")
    print(f"Target year                    : {TARGET_YEAR}")
    print(f"Total records in dataset       : {total_records}")
    print(f"Incidents in target year       : {incidents_in_year}")
    print(f"Count per attack type          : {dict(sorted(attack_counter.items()))}")
    print(f"Total financial loss (USD M)   : {total_loss}")
    print(f"Average financial loss (USD M) : {avg_loss}")
    print("Five costliest incidents:")
    if top_5_incidents:
        for rank, inc in enumerate(top_5_incidents, start=1):
            print(
                f"  {rank}. {inc.get('Country', 'N/A')} - {inc.get(COUNT_FIELD, 'N/A')} "
                f"(${inc.get(LOSS_FIELD, 'N/A')}M)"
            )
    else:
        print("  None found.")
    print(f"Filtered CSV file              : {FILTERED_CSV}")
    print(f"JSON summary file              : {SUMMARY_JSON}")
    print(f"Backup verification            : {'Verified (' + BACKUP_JSON + ')' if backup_verified else 'Failed'}")
    print(f"Report file                    : {REPORT_TXT}")


main()

=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===
Student                        : Solina (TUPM-26-2302)
Target year                    : 2017
Total records in dataset       : 3000
Incidents in target year       : 319
Count per attack type          : {'DDoS': 58, 'Malware': 46, 'Man-in-the-Middle': 58, 'Phishing': 64, 'Ransomware': 44, 'SQL Injection': 49}
Total financial loss (USD M)   : 16261.68
Average financial loss (USD M) : 50.98
Five costliest incidents:
  1. Australia - SQL Injection ($99.99M)
  2. Brazil - Malware ($99.22M)
  3. UK - SQL Injection ($98.75M)
  4. USA - DDoS ($98.73M)
  5. Germany - DDoS ($98.61M)
Filtered CSV file              : filtered_cybersecurity_2017.csv
JSON summary file              : cybersecurity_top5_2017.json
Backup verification            : Verified (cybersecurity_top5_2017_backup.json)
Report file                    : cybersecurity_report_2017.txt
